In [96]:
import pandas as pd
import numpy as np
import requests
from functools import reduce
import matplotlib.pyplot as plt
import pickle
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)
import sys
sys.path.append("../../Functions and Dictionaries") # Adds higher directory to python modules path
import CIPCodes
fam = CIPCodes.programfam
fam_inv = {v: k for k, v in fam.items()}
cat = CIPCodes.programcat
cat_inv = {v: k for k, v in cat.items()}
prog = CIPCodes.program
prog_inv = {v: k for k, v in prog.items()}
import geodict
tncountyfips = geodict.alltncountyfips

AttributeError: module 'geodict' has no attribute 'alltncountyfips'

In [65]:
oldest = pd.read_csv('../Data Downloads/TBR_TCAT_20192020_20212022.csv')
oldest.head(2)

,Total Awards,Award Type,Year,Institution,Campus,Six Digit CIP
0,2,Certificate,2019-20,Knoxville,Anderson County Career and Technical Center,480508
1,7,Diploma,2019-20,Knoxville,Anderson County Career and Technical Center,480508


In [66]:
older = pd.read_csv('../Data Downloads/TBR_TCAT_20222023.csv')
older.head(2)

,Total Awards,Institution,Award Type,Year,Campus,Six Digit CIP
0,12,Athens,Certificate,2022-23,Main Campus,480508
1,1,Athens,Certificate,2022-23,Main Campus,480501


In [67]:
current = pd.read_csv('../Data Downloads/TBR_TCAT_20232024.csv')
current.head(2)

,Total Awards,Institution,Award Type,Year,Campus,Six Digit CIP
0,10,Athens,Certificate,2023-24,Main Campus,480508
1,1,Athens,Certificate,2023-24,Main Campus,520402


In [68]:
dfs = [oldest, older, current]
data = pd.concat(dfs)

In [69]:
data.head()

,Total Awards,Award Type,Year,Institution,Campus,Six Digit CIP
0,2,Certificate,2019-20,Knoxville,Anderson County Career and Technical Center,480508
1,7,Diploma,2019-20,Knoxville,Anderson County Career and Technical Center,480508
2,3,Diploma,2019-20,Knoxville,Anderson County Career and Technical Center,480501
3,6,Diploma,2019-20,Knoxville,Oak Ridge High School,480508
4,10,Certificate,2020-21,Knoxville,Anderson County Career and Technical Center,480508


In [70]:
data.tail()

,Total Awards,Award Type,Year,Institution,Campus,Six Digit CIP
794,8,Diploma,2023-24,Upper Cumberland,Main Campus,470604
795,7,Diploma,2023-24,Upper Cumberland,Main Campus,470603
796,7,Diploma,2023-24,Upper Cumberland,Main Campus,480501
797,3,Diploma,2023-24,Upper Cumberland,Main Campus,470201
798,33,Diploma,2023-24,Upper Cumberland,Main Campus,513901


In [71]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3936 entries, 0 to 798
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Total Awards   3936 non-null   int64 
 1   Award Type     3936 non-null   object
 2   Year           3936 non-null   object
 3   Institution    3936 non-null   object
 4   Campus         3936 non-null   object
 5   Six Digit CIP  3936 non-null   int64 
dtypes: int64(2), object(4)
memory usage: 215.2+ KB


In [72]:
data['Six Digit CIP'] = data['Six Digit CIP'].astype(str)

In [73]:
data['Six Digit CIP'] = data['Six Digit CIP'].str.zfill(6)

In [74]:
data['Six Digit CIP'] = data['Six Digit CIP'].str.zfill(6).str.replace(r'^(\d{2})(\d+)$', r'\1.\2', regex=True)

In [75]:
boop = data['Six Digit CIP'].str.split("", n = 7, expand = True)
data['Four Digit CIP'] = boop[1]+boop[2]+boop[3]+boop[4]+boop[5]
data['Two Digit CIP'] = boop[1]+boop[2]
boop.head(2)

,0,1,2,3,4,5,6,7
0,,4,8,.,0,5,0,8
1,,4,8,.,0,5,0,8


In [76]:
data.head()

,Total Awards,Award Type,Year,Institution,Campus,Six Digit CIP,Four Digit CIP,Two Digit CIP
0,2,Certificate,2019-20,Knoxville,Anderson County Career and Technical Center,48.0508,48.05,48
1,7,Diploma,2019-20,Knoxville,Anderson County Career and Technical Center,48.0508,48.05,48
2,3,Diploma,2019-20,Knoxville,Anderson County Career and Technical Center,48.0501,48.05,48
3,6,Diploma,2019-20,Knoxville,Oak Ridge High School,48.0508,48.05,48
4,10,Certificate,2020-21,Knoxville,Anderson County Career and Technical Center,48.0508,48.05,48


In [79]:
data['Program Family'] = data['Two Digit CIP'].map(fam_inv)
data['Program Category'] = data['Four Digit CIP'].map(cat_inv)
data['Program'] = data['Six Digit CIP'].map(prog_inv)

In [80]:
data.head()

,Total Awards,Award Type,Year,Institution,Campus,Six Digit CIP,Four Digit CIP,Two Digit CIP,Program Family,Program Category,Program
0,2,Certificate,2019-20,Knoxville,Anderson County Career and Technical Center,48.0508,48.05,48,Precision Production,Precision Metal Working,Welding Technology/Welder
1,7,Diploma,2019-20,Knoxville,Anderson County Career and Technical Center,48.0508,48.05,48,Precision Production,Precision Metal Working,Welding Technology/Welder
2,3,Diploma,2019-20,Knoxville,Anderson County Career and Technical Center,48.0501,48.05,48,Precision Production,Precision Metal Working,Machine Tool Technology/Machinist
3,6,Diploma,2019-20,Knoxville,Oak Ridge High School,48.0508,48.05,48,Precision Production,Precision Metal Working,Welding Technology/Welder
4,10,Certificate,2020-21,Knoxville,Anderson County Career and Technical Center,48.0508,48.05,48,Precision Production,Precision Metal Working,Welding Technology/Welder


In [88]:
soc = pd.read_csv('../Data Downloads/CrosswalkCIPSOC.csv', dtype = str)
soc.drop(columns = 'Program', inplace = True)

In [89]:
soc.head()

,Six Digit CIP,6 Digit SOC,Occupation
0,01.0000,19-1011,Animal Scientists
1,01.0000,19-1012,Food Scientists and Technologists
2,01.0000,19-1013,Soil and Plant Scientists
3,01.0000,19-4012,Agricultural Technicians
4,01.0000,25-1041,"Agricultural Sciences Teachers, Postsecondary"


In [90]:
test = data.merge(soc, on = 'Six Digit CIP')

In [92]:
test.tail()

,Total Awards,Award Type,Year,Institution,Campus,Six Digit CIP,Four Digit CIP,Two Digit CIP,Program Family,Program Category,Program,6 Digit SOC,Occupation
13840,6,Certificate,2023-24,Elizabethton,Unaka High School,12.0506,12.05,12,"Culinary, Entertainment, and Personal Services",Culinary Arts and Related Services,Meat Cutting/Meat Cutter,51-3021,Butchers and Meat Cutters
13841,6,Certificate,2023-24,Elizabethton,Unaka High School,12.0506,12.05,12,"Culinary, Entertainment, and Personal Services",Culinary Arts and Related Services,Meat Cutting/Meat Cutter,51-3023,Slaughterers and Meat Packers
13842,5,Diploma,2023-24,Pulaski,Lawrence County Instructional Service Center,48.0510,48.05,48,Precision Production,Precision Metal Working,Computer Numerically Controlled (CNC) Machinis...,51-9161,Computer Numerically Controlled Tool Operators
13843,5,Diploma,2023-24,Pulaski,Lawrence County Instructional Service Center,48.0510,48.05,48,Precision Production,Precision Metal Working,Computer Numerically Controlled (CNC) Machinis...,51-9162,Computer Numerically Controlled Tool Programmers
13844,13,Certificate,2023-24,Shelbyville,Lincoln Central Academy,51.0000,51.00,51,Health Professions and Related Programs,"Health Services/Allied Health/Health Sciences,...","Health Services/Allied Health/Health Sciences,...",99-9999,NO MATCH
